In [2]:
!pip install pyswarms
!pip install scikit-learn


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\Benjo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\Benjo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
import pyswarms as ps
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import time

iris = load_iris()
X = iris.data
y_labels = iris.target

n_classes_iris = 3
y_one_hot = np.zeros((y_labels.shape[0], n_classes_iris))
for i in range(y_labels.shape[0]):
    y_one_hot[i, y_labels[i]] = 1.0

X_train, X_test, y_train, y_test = train_test_split(X, y_one_hot, test_size=0.3, random_state=42)

num_samples_train = X_train.shape[0]
num_samples_test = X_test.shape[0]

n_inputs = X_train.shape[1] 
n_hidden_neurons = 20       

dimensions = (n_inputs * n_hidden_neurons) + \
             n_hidden_neurons + \
             (n_hidden_neurons * n_classes_iris) + \
             n_classes_iris

print(f"Neural Network Parameters to Optimize (PSO dimensions): {dimensions}") 


options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
n_particles_pso = 100
iters_pso = 100      

min_bound_val = -5.0
max_bound_val = 5.0
min_bounds = np.full(dimensions, min_bound_val)
max_bounds = np.full(dimensions, max_bound_val)
pso_bounds = (min_bounds, max_bounds)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def tanh_activation(z): 
    return np.tanh(z)

def softmax(logits):
    """Computes softmax probabilities from logits."""
    exp_scores = np.exp(logits - np.max(logits, axis=1, keepdims=True)) 
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

def get_nn_parameters(particle_params):
    """
    Reshapes a flat particle (1D array of parameters) into
    neural network weights (W1, W2) and biases (b1, b2).
    """
    W1_end = n_inputs * n_hidden_neurons
    b1_end = W1_end + n_hidden_neurons
    W2_end = b1_end + (n_hidden_neurons * n_classes_iris)

    W1 = particle_params[0:W1_end].reshape((n_inputs, n_hidden_neurons))
    b1 = particle_params[W1_end:b1_end].reshape((n_hidden_neurons,))
    W2 = particle_params[b1_end:W2_end].reshape((n_hidden_neurons, n_classes_iris))
    b2 = particle_params[W2_end:].reshape((n_classes_iris,))
    
    return W1, b1, W2, b2

def probs_function(particle_params, X_data):
    """
    Performs a forward pass and returns probabilities.
    'particle_params' is a single particle (1D array of NN weights/biases).
    'X_data' is the input feature matrix.
    """
    W1, b1, W2, b2 = get_nn_parameters(particle_params)

    z1 = X_data.dot(W1) + b1
    a1 = tanh_activation(z1) 
    logits = a1.dot(W2) + b2
    probabilities = softmax(logits)
    
    return probabilities

def categorical_cross_entropy_loss(y_true, y_pred_probs):
    """
    Computes categorical cross-entropy loss.
    y_true: one-hot encoded true labels
    y_pred_probs: predicted probabilities from softmax
    """
    epsilon = 1e-12
    y_pred_probs = np.clip(y_pred_probs, epsilon, 1. - epsilon)
    
    loss = -np.sum(y_true * np.log(y_pred_probs)) / y_true.shape[0] 
    return loss

def forward_prop_for_pso(particle_params):
    """
    Calculates the loss for a single particle on the training data.
    This function will be called by the swarm objective function 'f'.
    """
    probabilities_train = probs_function(particle_params, X_train)
    
    loss = categorical_cross_entropy_loss(y_train, probabilities_train)
    return loss

def f(particles_matrix):
    """
    Objective function for PSO.
    'particles_matrix' is a 2D NumPy array where each row is a particle's parameters.
    It computes the loss for each particle in the swarm.
    """
    n_swarm_particles = particles_matrix.shape[0]
    losses = np.zeros(n_swarm_particles) 
    
    for i in range(n_swarm_particles):
        losses[i] = forward_prop_for_pso(particles_matrix[i])
        
    return losses


# --- 4. Initialize and Run PSO ---
print("\nInitializing PSO optimizer...")
optimizer = ps.single.GlobalBestPSO(n_particles=n_particles_pso,
                                    dimensions=dimensions,
                                    options=options,
                                    bounds=pso_bounds,
                                    )

print("Starting PSO optimization...")
start_pso_time = time.time()
best_cost, best_pos = optimizer.optimize(f, iters=iters_pso, verbose=True)
end_pso_time = time.time()
pso_duration = end_pso_time - start_pso_time

print(f"PSO optimization finished in {pso_duration:.2f} seconds.")
print(f"Best cost (min loss) found by PSO: {best_cost:.4f}")

def predict_nn(optimized_params, X_data_to_predict):
    """
    Makes predictions using the optimized network parameters.
    """
    probabilities = probs_function(optimized_params, X_data_to_predict)
    predictions = np.argmax(probabilities, axis=1)
    return predictions

y_test_pred_labels = predict_nn(best_pos, X_test)

y_test_true_labels = np.argmax(y_test, axis=1)

accuracy = np.mean(y_test_pred_labels == y_test_true_labels)
print(f"\nAccuracy on the test set: {accuracy * 100:.2f}%")

print("\nSample predictions vs actual (test set):")
for i in range(min(10, num_samples_test)): 
    print(f"Predicted: {iris.target_names[y_test_pred_labels[i]]}, Actual: {iris.target_names[y_test_true_labels[i]]}")

2025-06-03 23:15:56,238 - pyswarms.single.global_best - INFO - Optimize for 100 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


Neural Network Parameters to Optimize (PSO dimensions): 163

Initializing PSO optimizer...
Starting PSO optimization...


pyswarms.single.global_best: 100%|██████████|100/100, best_cost=0.24
2025-06-03 23:15:56,702 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.2399607780391093, best pos: [ 1.8779113  -0.6628797   3.86687271  1.25228867 -0.75596966  0.74222346
  0.56823487  1.14489808 -0.27410234 -2.69769076  3.27040276  0.59630408
 -4.76612558  3.91018035  0.42962205 -0.53049494  2.68671442 -3.85814023
 -2.03019658 -0.13268541  1.92188251  1.48624239  1.82859395 -1.30181033
  1.93277324  1.60524103  2.33411335  3.66428687  3.17080447 -0.51826783
 -0.12076609 -1.81432331  0.46712972  2.06918978  1.0088086   2.38187919
  2.74834263  1.56649685  0.54322331 -0.51277105 -3.28042935 -1.41897838
  0.21174212 -3.03469332  0.16498509 -1.34697096 -1.94965901 -0.99731576
 -1.89835971 -0.75704484 -2.04920397  1.83073821 -2.32878849  2.34278259
  1.00800269 -1.68918698 -0.45692772  2.78083172  0.59543429 -0.95550585
 -1.05970216  0.84096666  0.97628345 -3.93263874  0.57885779  2.38416261


PSO optimization finished in 0.47 seconds.
Best cost (min loss) found by PSO: 0.2400

Accuracy on the test set: 86.67%

Sample predictions vs actual (test set):
Predicted: versicolor, Actual: versicolor
Predicted: setosa, Actual: setosa
Predicted: virginica, Actual: virginica
Predicted: versicolor, Actual: versicolor
Predicted: versicolor, Actual: versicolor
Predicted: setosa, Actual: setosa
Predicted: setosa, Actual: versicolor
Predicted: versicolor, Actual: virginica
Predicted: virginica, Actual: versicolor
Predicted: versicolor, Actual: versicolor
